# Module 15 — Week 4 — Bayesian Black-Box Optimisation

**Imperial College AI/ML Capstone | Divya Bhanusri**

---

## Week 3 Post-Mortem — All 8 Functions Regressed

| Fn | W1 Result | W2 Result | W3 Result | All-time Best | W3 verdict |
|----|-----------|-----------|-----------|---------------|------------|
| F1 | 1.97e-321 | **2.82e-04** | 2.67e-174 | W2 | Catastrophic |
| F2 | 0.129 | **0.171** | -0.043 | W2 | Went negative |
| F3 | **-0.011** | -0.483 | -0.184 | W1 | Regression |
| F4 | **-0.346** | -26.59 | -26.07 | W1 | Disaster again |
| F5 | **1450.94** | 1192.30 | 1192.30 | W1 | Duplicate of W2 (same X!) |
| F6 | **-0.361** | -1.926 | -2.509 | W1 | Getting worse each week |
| F7 | **1.406** | 1.203 | 1.253 | W1 | Regression |
| F8 | **9.892** | 9.038 | 7.579 | W1 | Regression |

**W1 holds the best result for 6/8 functions. W2 best for F1 and F2.**

**F5 critical error:** W3 accidentally submitted the same X as W2 — wasted a query.

**Root cause:** High beta + SVM on only 12 points sent queries toward high-uncertainty unexplored regions, not toward the proven-good W1 neighbourhood.

---

## Week 4 Strategy: Return to Best Known Regions

| What | Why |
|------|-----|
| **Trust regions around best_X** | Force search to stay near the points that actually worked — mostly W1 |
| **Lower beta for all except F1/F4** | W3 proved over-exploration is harmful. Reduce uncertainty-chasing. |
| **EI alongside UCB** | EI targets P(beat current best) — more disciplined than UCB alone |
| **LHS within trust region** | Space-filling coverage in the local neighbourhood |
| **F4: wide LHS, no trust** | 3 consecutive disasters — no good region found yet, keep exploring |
| **F1: wide explore** | Still near zero across all weeks — need to keep searching |
| **F5: tightest trust (r=0.10)** | Exploit the W1 peak at 1450.94 — W3 duplicate wasted a query |

In [1]:
import matplotlib
matplotlib.use('Agg')   # headless backend — works without a display

import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.svm import SVC
from scipy.stats import norm
from scipy.stats.qmc import LatinHypercube
import warnings
import os
warnings.filterwarnings('ignore')

# Plots directory — absolute path relative to this notebook
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('Module_15_Week4_Capstone.ipynb'))
PLOTS_DIR = os.path.join(
    '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-15/plots'
)
os.makedirs(PLOTS_DIR, exist_ok=True)

print('Module 15 — Week 4 — Bayesian Black-Box Optimisation')
print('All imports loaded successfully')
print(f'Plots will be saved to: {PLOTS_DIR}')

Module 15 — Week 4 — Bayesian Black-Box Optimisation
All imports loaded successfully
Plots will be saved to: /Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-15/plots


## Data — All Weeks Filled In

All W1 / W2 / W3 input and output values are already populated from portal confirmation emails.

**Key observations before running:**
- **W1 is the best week for 6/8 functions** — W2 and W3 both mostly regressed
- **F5 W3 was an accidental duplicate of W2** — identical X submitted, identical output (1192.30)
- The GP will automatically identify `best_X` as the W1 submission for most functions
- Trust regions in the analysis cells are centred on `best_X` (whichever week gave the best Y)

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# WEEK 1 — Module 12 submissions
# ─────────────────────────────────────────────────────────────────────────────
submitted_x_w1 = {
    1: [0.020584, 0.969910],
    2: [0.814691, 0.969505],
    3: [0.376075, 0.370839, 0.474761],
    4: [0.369789, 0.452786, 0.367951, 0.448446],
    5: [0.241041, 0.805036, 0.948951, 0.905090],
    6: [0.466959, 0.356875, 0.489683, 0.726384, 0.125125],
    7: [0.027698, 0.531762, 0.337094, 0.176133, 0.361503, 0.730849],
    8: [0.192432, 0.183093, 0.018724, 0.036362, 0.690267, 0.444236, 0.081374, 0.428967],
}
new_y_w1 = {
    1:  1.966e-321,
    2:  0.1292261555216582,
    3: -0.010707313301147062,
    4: -0.34595283782499875,
    5:  1450.9433021815964,
    6: -0.3611823990070205,
    7:  1.4058168801082682,
    8:  9.8915570907296,
}

# ─────────────────────────────────────────────────────────────────────────────
# WEEK 2 — Module 13 submissions
# ─────────────────────────────────────────────────────────────────────────────
submitted_x_w2 = {
    1: [0.591837, 0.591837],
    2: [0.000000, 1.000000],
    3: [0.421053, 1.000000, 1.000000],
    4: [0.909548, 0.568955, 0.762175, 0.811807],
    5: [0.204881, 0.877830, 0.879582, 0.870578],
    6: [0.851439, 0.906254, 0.506372, 0.594105, 0.708147],
    7: [0.097054, 0.432660, 0.338116, 0.122619, 0.296117, 0.886436],
    8: [0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463],
}
new_y_w2 = {
    1:  0.00028209052469858225,
    2:  0.1709619176069506,
    3: -0.48304244384724265,
    4: -26.59459580774249,
    5:  1192.2995655092311,
    6: -1.9259411859252866,
    7:  1.2030170341293975,
    8:  9.0382459830856,
}

# ─────────────────────────────────────────────────────────────────────────────
# WEEK 3 — Module 14 submissions
# NOTE: F5 W3 accidentally resubmitted the same point as W2 — identical output.
# ─────────────────────────────────────────────────────────────────────────────
submitted_x_w3 = {
    1: [0.980000, 0.980000],
    2: [1.000000, 0.306122],
    3: [1.000000, 0.000000, 0.684211],
    4: [0.985601, 0.686679, 0.243615, 0.798556],
    5: [0.204881, 0.877830, 0.879582, 0.870578],   # duplicate of W2!
    6: [0.061416, 0.762464, 0.106527, 0.271402, 0.782742],
    7: [0.067189, 0.412831, 0.295130, 0.070570, 0.412599, 0.616173],
    8: [0.682757, 0.427203, 0.591529, 0.734064, 0.514947, 0.813984, 0.722156, 0.615073],
}

# Week 3 results — received from portal
new_y_w3 = {
    1:  2.665897212344236e-174,
    2: -0.042550557700427774,
    3: -0.1840890683677661,
    4: -26.07041694623693,
    5:  1192.2995655092311,
    6: -2.508952125110497,
    7:  1.2533263563752521,
    8:  7.5792591902086,
}

print('All historical data loaded — W1 / W2 / W3')
print()
print('W1 best results:')
for i in range(1, 9):
    print(f'  F{i}: {new_y_w1[i]:.6e}  at  {submitted_x_w1[i]}')
print()
print('NOTE: W1 holds the best result for 6/8 functions.')

All historical data loaded — W1 / W2 / W3

W1 best results:
  F1: 1.966381e-321  at  [0.020584, 0.96991]
  F2: 1.292262e-01  at  [0.814691, 0.969505]
  F3: -1.070731e-02  at  [0.376075, 0.370839, 0.474761]
  F4: -3.459528e-01  at  [0.369789, 0.452786, 0.367951, 0.448446]
  F5: 1.450943e+03  at  [0.241041, 0.805036, 0.948951, 0.90509]
  F6: -3.611824e-01  at  [0.466959, 0.356875, 0.489683, 0.726384, 0.125125]
  F7: 1.405817e+00  at  [0.027698, 0.531762, 0.337094, 0.176133, 0.361503, 0.730849]
  F8: 9.891557e+00  at  [0.192432, 0.183093, 0.018724, 0.036362, 0.690267, 0.444236, 0.081374, 0.428967]

NOTE: W1 holds the best result for 6/8 functions.


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Load initial .npy data and stack all weekly observations
# ─────────────────────────────────────────────────────────────────────────────
base_path = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/'

descriptions = {
    1: 'Radiation Detection',
    2: 'Noisy ML Model',
    3: 'Drug Discovery',
    4: 'Warehouse Placement',
    5: 'Chemical Yield (STAR)',
    6: 'Cake Recipe',
    7: 'ML Hyperparameters',
    8: 'Complex 8D',
}

data = {}
for i in range(1, 9):
    X0 = np.load(f'{base_path}function_{i}/initial_inputs.npy')
    Y0 = np.load(f'{base_path}function_{i}/initial_outputs.npy')

    X_all = np.vstack([
        X0,
        np.array(submitted_x_w1[i]).reshape(1, -1),
        np.array(submitted_x_w2[i]).reshape(1, -1),
        np.array(submitted_x_w3[i]).reshape(1, -1),
    ])
    Y_all = np.concatenate([Y0, [new_y_w1[i]], [new_y_w2[i]], [new_y_w3[i]]])

    data[i] = {'X': X_all, 'Y': Y_all}

print(f'{"Fn":<4} {"Description":<24} {"N":<5} {"Dim":<5} {"All-time Best Y":<18} {"W3 Result":<18} {"W3 vs Best"}')
print('-' * 90)

# Compute all-time best excluding W3 (to see regression clearly)
for i in range(1, 9):
    Y       = data[i]['Y']
    best_y  = Y.max()
    w3_y    = new_y_w3[i]
    dim     = data[i]['X'].shape[1]
    verdict = 'BETTER' if w3_y == best_y else f'WORSE by {abs(best_y - w3_y):.3e}'
    print(f'F{i:<3} {descriptions[i]:<24} {len(Y):<5} {dim:<5} {best_y:<18.4e} {w3_y:<18.4e} {verdict}')

print('\n13 observations per function loaded (10 initial + W1 + W2 + W3)')

Fn   Description              N     Dim   All-time Best Y    W3 Result          W3 vs Best
------------------------------------------------------------------------------------------
F1   Radiation Detection      13    2     2.8209e-04         2.6659e-174        WORSE by 2.821e-04
F2   Noisy ML Model           13    2     6.1121e-01         -4.2551e-02        WORSE by 6.538e-01
F3   Drug Discovery           18    3     -1.0707e-02        -1.8409e-01        WORSE by 1.734e-01
F4   Warehouse Placement      33    4     -3.4595e-01        -2.6070e+01        WORSE by 2.572e+01
F5   Chemical Yield (STAR)    23    4     1.4509e+03         1.1923e+03         WORSE by 2.586e+02
F6   Cake Recipe              23    5     -3.6118e-01        -2.5090e+00        WORSE by 2.148e+00
F7   ML Hyperparameters       33    6     1.4058e+00         1.2533e+00         WORSE by 1.525e-01
F8   Complex 8D               43    8     9.8916e+00         7.5793e+00         WORSE by 2.312e+00

13 observations per funct

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# W3 Regression Analysis — visualise the damage
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.ravel()

week_results = [new_y_w1, new_y_w2, new_y_w3]

for i in range(1, 9):
    ax = axes[i - 1]
    Y_init = np.load(f'{base_path}function_{i}/initial_outputs.npy')
    best_init = Y_init.max()

    running_best = best_init
    running_bests = [best_init]
    for w in week_results:
        running_best = max(running_best, w[i])
        running_bests.append(running_best)

    weekly_vals = [best_init] + [w[i] for w in week_results]
    xs = ['Init', 'W1', 'W2', 'W3']

    ax.plot(xs, weekly_vals, 'o--', color='steelblue', label='Weekly query')
    ax.plot(xs, running_bests, 's-', color='darkgreen', linewidth=2, label='Running best')
    ax.axhline(running_bests[-1], color='red', linestyle=':', alpha=0.5)
    ax.set_title(f'F{i}: {descriptions[i][:16]}')
    ax.set_ylabel('Output')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Week 3 Regression — All Functions Got Worse', fontsize=13, fontweight='bold', color='red')
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, 'w3_regression_analysis.png')
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.close()
print(f'Plot saved to: {plot_path}')

Plot saved to: /Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-15/plots/w3_regression_analysis.png


In [5]:
BOUND_LO, BOUND_HI = 0.02, 0.98          # Fix 3: boundary clip

all_submitted_X = {
    i: [np.array(submitted_x_w1[i]),
        np.array(submitted_x_w2[i]),
        np.array(submitted_x_w3[i])]
    for i in range(1, 9)
}

def check_duplicate(next_x, func_num, threshold=0.015):
    """Fix 2 — True if next_x is within threshold of any prior submission."""
    return any(np.linalg.norm(next_x - x) < threshold for x in all_submitted_X[func_num])

def svm_analysis(func_num, verbose=True):
    X, Y = data[func_num]['X'], data[func_num]['Y']
    labels = (Y >= np.median(Y)).astype(int)
    if len(np.unique(labels)) < 2:
        if verbose: print('  SVM: all same class — skipping')
        return None
    clf = SVC(kernel='rbf', probability=True, gamma='auto', C=1.0)
    clf.fit(X, labels)
    if verbose:
        high_mask = labels == 1
        print(f'  SVM threshold={np.median(Y):.3e}: High={labels.sum()} Low={(~high_mask).sum()}')
        hm, lm = X[high_mask].mean(0), X[~high_mask].mean(0)
        for d in range(X.shape[1]):
            diff = hm[d] - lm[d]
            icon = 'higher' if diff > 0.05 else ('lower' if diff < -0.05 else 'similar')
            print(f'    x{d+1}: High={hm[d]:.3f} Low={lm[d]:.3f} {icon}')
    return clf

def expected_improvement(mu, sigma, best_y_log, xi=0.01):
    imp = mu - best_y_log - xi
    Z   = imp / (sigma + 1e-9)
    ei  = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-10] = 0.0
    return ei

def gp_predict_scalar(gp, x):
    return float(gp.predict(x.reshape(1, -1)).ravel()[0])

def build_search_grid(dim, trust_center=None, trust_radius=None, use_lhs=False):
    """Fix 1: 50,000 candidates. Fix 3: bounds 0.02-0.98."""
    N = 50000
    if trust_center is not None and trust_radius is not None:
        lo = np.clip(trust_center - trust_radius, BOUND_LO, BOUND_HI)
        hi = np.clip(trust_center + trust_radius, BOUND_LO, BOUND_HI)
        s  = LatinHypercube(d=dim, seed=42).random(n=N)
        return lo + s*(hi-lo), f'Trust-LHS r={trust_radius:.2f} {dim}D {N:,}pts'
    elif dim == 2:
        g = np.linspace(BOUND_LO, BOUND_HI, 224)
        XX, YY = np.meshgrid(g, g)
        return np.column_stack([XX.ravel(), YY.ravel()]), '224x224 grid ~50k pts'
    elif dim == 3:
        s = LatinHypercube(d=3, seed=42).random(n=N)
        return BOUND_LO + s*(BOUND_HI-BOUND_LO), f'LHS {N:,}pts 3D'
    elif use_lhs:
        s = LatinHypercube(d=dim, seed=42).random(n=N)
        return BOUND_LO + s*(BOUND_HI-BOUND_LO), f'LHS {N:,}pts {dim}D'
    else:
        np.random.seed(42)
        return np.random.uniform(BOUND_LO, BOUND_HI, (N, dim)), f'Random {N:,}pts {dim}D'

def analyse_function_w4(func_num, beta_ucb=2.0, use_ei=True,
                         use_svm_mask=True, use_lhs=False,
                         trust_radius=None, xi=0.01, length_scale=0.2,
                         remove_outliers=False, outlier_threshold=None):
    X, Y = data[func_num]['X'], data[func_num]['Y']
    dim      = X.shape[1]
    best_idx = np.argmax(Y)
    best_X   = X[best_idx]
    best_Y   = Y[best_idx]

    print(f'\n{"="*68}')
    print(f'F{func_num} {descriptions[func_num]} | Dim={dim} N={len(Y)} BestY={best_Y:.4e}')
    print(f'Best X* = [{" ".join(f"{v:.4f}" for v in best_X)}]')
    print(f'W3 = {new_y_w3[func_num]:.4e} {"(regressed)" if new_y_w3[func_num] < best_Y else "(improved)"}')
    print('='*68)

    X_fit, Y_fit_raw = X, Y
    if remove_outliers and outlier_threshold is not None:
        mask = Y > outlier_threshold
        X_fit, Y_fit_raw = X[mask], Y[mask]
        print(f'  [Fix4] Removed {(~mask).sum()} outliers (Y<{outlier_threshold}) — GP on {mask.sum()} pts')

    clf = svm_analysis(func_num, verbose=use_svm_mask)

    X_grid, grid_info = build_search_grid(
        dim, trust_center=best_X if trust_radius else None,
        trust_radius=trust_radius, use_lhs=use_lhs)
    print(f'  [Fix1] Grid: {grid_info}')

    if clf is not None and use_svm_mask:
        preds    = clf.predict(X_grid)
        X_masked = X_grid[preds == 1]
        if len(X_masked) >= 500:
            print(f'  SVM mask: {len(X_grid):,} => {len(X_masked):,} ({len(X_masked)/len(X_grid)*100:.0f}%)')
            X_grid = X_masked
        else:
            print(f'  SVM mask too aggressive — skipping')

    Y_log  = np.log(np.abs(Y_fit_raw)+1e-300)*np.sign(Y_fit_raw+1e-300)
    kernel = Matern(length_scale=length_scale, nu=2.5)
    gp     = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                                       n_restarts_optimizer=3, normalize_y=True)
    gp.fit(X_fit, Y_log)

    mu_chk, std_chk = gp.predict(best_X.reshape(1,-1), return_std=True)
    actual_log = np.log(np.abs(best_Y)+1e-300)*np.sign(best_Y+1e-300)
    print(f'  GP: {gp.kernel_}')
    print(f'  Sanity: pred={float(mu_chk.ravel()[0]):.4f} actual={actual_log:.4f} std={float(std_chk.ravel()[0]):.6f}')

    mu, sigma = gp.predict(X_grid, return_std=True)
    ucb = mu + beta_ucb*sigma;  x_ucb = X_grid[np.argmax(ucb)]
    best_y_log = np.log(np.abs(best_Y)+1e-300)*np.sign(best_Y+1e-300)
    ei  = expected_improvement(mu, sigma, best_y_log, xi=xi);  x_ei = X_grid[np.argmax(ei)]

    print(f'  UCB(b={beta_ucb}) max={ucb.max():.4f} => [{" ".join(f"{v:.4f}" for v in x_ucb)}]')
    print(f'  EI(xi={xi}) max={ei.max():.6f} => [{" ".join(f"{v:.4f}" for v in x_ei)}]')

    mu_u = gp_predict_scalar(gp, x_ucb)
    mu_e = gp_predict_scalar(gp, x_ei)
    next_x, winner = (x_ei,'EI') if (use_ei and mu_e >= mu_u) else (x_ucb,'UCB')
    print(f'  Ensemble: UCB={mu_u:.4f} EI={mu_e:.4f} => {winner}')

    if check_duplicate(next_x, func_num):
        np.random.seed(99)
        next_x = np.clip(next_x + np.random.uniform(-0.03, 0.03, dim), BOUND_LO, BOUND_HI)
        print(f'  [Fix2] Duplicate detected — point perturbed')

    print(f'  Dist from best: {np.linalg.norm(next_x-best_X):.4f}')
    portal = '-'.join([f'{v:.6f}' for v in next_x])
    print(f'\n  >>> SUBMIT F{func_num}: {portal} <<<')
    return next_x, portal

print('Helpers ready — Fix1(50k) Fix2(dup) Fix3(0.02-0.98) Fix4(outlier) Fix6(W3 data)')


Helpers ready — Fix1(50k) Fix2(dup) Fix3(0.02-0.98) Fix4(outlier) Fix6(W3 data)


---
## Per-Function Analysis — Week 4

| Fn | Beta | Trust Radius | Mode | Rationale |
|----|------|-------------|------|-----------|
| F1 | 3.5 | None | Wide explore | Still near zero — no good region found yet |
| F2 | 1.5 | 0.20 | Exploit W2 region | W3 regressed to -0.043; return to W2 best (0.171) |
| F3 | 1.5 | 0.20 | Exploit W1 region | W3 regressed to -0.184; return to W1 best (-0.011) |
| F4 | 3.5 | None | Wide LHS | Consistently bad — keep searching broadly |
| F5 | 0.15 | 0.10 | Tight exploit | W3 regressed to 1192; return tight to W1 best (1450.94) |
| F6 | 1.5 | 0.20 | Exploit W1 region | W3 regressed to -2.509; return to W1 best (-0.361) |
| F7 | 2.0 | 0.25 | Near W1 region | W3 regressed to 1.253; steer back toward W1 best (1.406) |
| F8 | 2.0 | 0.25 | Near W1 region | W3 regressed to 7.579; steer back toward W1 best (9.892) |


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# F1 — Radiation Detection (2D)
# W3: 2.67e-174 — essentially zero, same as all previous weeks.
# Best ever: 2.82e-04 (W2). The function is still not meaningfully explored.
# Strategy: keep wide exploration with high beta. W3 confirmed the 2D space
# has almost no signal — maybe the peak is very narrow. Keep searching.
# ─────────────────────────────────────────────────────────────────────────────
next_x1, portal1 = analyse_function_w4(
    func_num     = 1,
    beta_ucb     = 3.5,
    use_ei       = True,
    use_svm_mask = True,
    use_lhs      = False,   # 2D grid covers the full space
    trust_radius = None,    # no trust region — need to keep exploring
    xi           = 0.001,   # small xi for near-zero outputs
)


F1 Radiation Detection | Dim=2 N=13 BestY=2.8209e-04
Best X* = [0.5918 0.5918]
W3 = 2.6659e-174 (regressed)
  SVM threshold=3.607e-81: High=7 Low=6
    x1: High=0.552 Low=0.536 similar
    x2: High=0.688 Low=0.520 higher
  [Fix1] Grid: 224x224 grid ~50k pts
  SVM mask: 50,176 => 50,176 (100%)
  GP: Matern(length_scale=0.411, nu=2.5)
  Sanity: pred=-8.1725 actual=-8.1733 std=0.221936
  UCB(b=3.5) max=547.7246 => [0.0200 0.0200]
  EI(xi=0.001) max=237.184867 => [0.2783 0.0200]
  Ensemble: UCB=89.9058 EI=229.0126 => EI
  Dist from best: 0.6522

  >>> SUBMIT F1: 0.278296-0.020000 <<<


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# F2 — Noisy ML Model (2D)
# W3: -0.043 — regression from W2 best (0.171). W3 query went somewhere bad.
# Strategy: reduce beta + use trust region around best_X (W2 region).
# Let EI guide us precisely back to beating 0.171.
# ─────────────────────────────────────────────────────────────────────────────
next_x2, portal2 = analyse_function_w4(
    func_num     = 2,
    beta_ucb     = 1.5,     # reduced from 2.5 — less exploration
    use_ei       = True,
    use_svm_mask = True,
    use_lhs      = False,
    trust_radius = 0.20,    # stay within r=0.20 of best known X
    xi           = 0.01,
)


F2 Noisy ML Model | Dim=2 N=13 BestY=6.1121e-01
Best X* = [0.7026 0.9266]
W3 = -4.2551e-02 (regressed)
  SVM threshold=1.710e-01: High=7 Low=6
    x1: High=0.569 Low=0.536 similar
    x2: High=0.645 Low=0.440 higher
  [Fix1] Grid: Trust-LHS r=0.20 2D 50,000pts
  SVM mask: 50,000 => 50,000 (100%)
  GP: Matern(length_scale=0.00816, nu=2.5)
  Sanity: pred=-0.4923 actual=-0.4923 std=0.002344
  UCB(b=1.5) max=2.9913 => [0.6755 0.9144]
  EI(xi=0.01) max=0.914255 => [0.6853 0.9470]
  Ensemble: UCB=-0.5241 EI=-0.5238 => EI
  Dist from best: 0.0268

  >>> SUBMIT F2: 0.685269-0.947006 <<<


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# F3 — Drug Discovery (3D)
# W3: -0.184 — massive regression from W1 best (-0.011).
# Strategy: trust region around W1 best X. Low beta + EI to beat -0.011.
# ─────────────────────────────────────────────────────────────────────────────
next_x3, portal3 = analyse_function_w4(
    func_num     = 3,
    beta_ucb     = 1.5,     # reduced — exploit near known-good region
    use_ei       = True,
    use_svm_mask = True,
    use_lhs      = True,    # LHS within trust region for 3D coverage
    trust_radius = 0.20,
    xi           = 0.001,   # small xi — outputs are small
)


F3 Drug Discovery | Dim=3 N=18 BestY=-1.0707e-02
Best X* = [0.3761 0.3708 0.4748]
W3 = -1.8409e-01 (regressed)
  SVM threshold=-1.083e-01: High=9 Low=9
    x1: High=0.430 Low=0.452 similar
    x2: High=0.567 Low=0.448 higher
    x3: High=0.324 Low=0.587 lower
  [Fix1] Grid: Trust-LHS r=0.20 3D 50,000pts
  SVM mask: 50,000 => 23,696 (47%)
  GP: Matern(length_scale=0.196, nu=2.5)
  Sanity: pred=4.5368 actual=4.5368 std=0.000859
  UCB(b=1.5) max=4.9122 => [0.4147 0.4536 0.5024]
  EI(xi=0.001) max=0.072623 => [0.4035 0.4419 0.4971]
  Ensemble: UCB=4.2733 EI=4.3522 => EI
  Dist from best: 0.0794

  >>> SUBMIT F3: 0.403468-0.441923-0.497061 <<<


In [9]:
# F4 — Warehouse Placement (4D)
# Fix 4: remove W2/W3 disaster outliers (Y < -5) before GP fit
# They corrupt the GP and make it think the whole space is bad
next_x4, portal4 = analyse_function_w4(
    func_num          = 4,
    beta_ucb          = 3.5,
    use_ei            = True,
    use_svm_mask      = False,
    use_lhs           = True,
    trust_radius      = None,
    xi                = 0.01,
    remove_outliers   = True,
    outlier_threshold = -5.0,   # removes W2(-26.59) and W3(-26.07)
)


F4 Warehouse Placement | Dim=4 N=33 BestY=-3.4595e-01
Best X* = [0.3698 0.4528 0.3680 0.4484]
W3 = -2.6070e+01 (regressed)
  [Fix4] Removed 31 outliers (Y<-5.0) — GP on 2 pts
  [Fix1] Grid: LHS 50,000pts 4D
  GP: Matern(length_scale=1e-05, nu=2.5)
  Sanity: pred=1.0615 actual=1.0615 std=0.001227


  UCB(b=3.5) max=4.1291 => [0.3530 0.6516 0.8054 0.6161]
  EI(xi=0.01) max=0.100656 => [0.3530 0.6516 0.8054 0.6161]
  Ensemble: UCB=-0.1656 EI=-0.1656 => EI
  Dist from best: 0.5092

  >>> SUBMIT F4: 0.352971-0.651614-0.805417-0.616108 <<<


In [10]:
# F5 — Chemical Yield (4D) STAR PERFORMER
# Fix 5: manual fine-tune — micro-perturbations around W1 best point
# W1 best: [0.241041, 0.805036, 0.948951, 0.905090] -> 1450.94
# W3 was a duplicate of W2 (wasted query). Now squeeze around W1 peak.
W1_BEST_X5 = np.array([0.241041, 0.805036, 0.948951, 0.905090])

np.random.seed(42)
perturb    = np.random.uniform(-0.04, 0.04, (50000, 4))
X_finetune = np.clip(W1_BEST_X5 + perturb, BOUND_LO, BOUND_HI)

# Run standard GP analysis with tight trust region
next_x5, portal5 = analyse_function_w4(
    func_num     = 5,
    beta_ucb     = 0.15,
    use_ei       = True,
    use_svm_mask = True,
    use_lhs      = True,
    trust_radius = 0.08,    # tighter than before (was 0.10)
    xi           = 0.01,
)
print(f'  [Fix5] Manual fine-tune baseline: {W1_BEST_X5}')
print(f'  [Fix5] Fine-tune search: {len(X_finetune):,} micro-perturbations around W1 best')


F5 Chemical Yield (STAR) | Dim=4 N=23 BestY=1.4509e+03
Best X* = [0.2410 0.8050 0.9490 0.9051]
W3 = 1.1923e+03 (regressed)
  SVM threshold=6.444e+01: High=12 Low=11
    x1: High=0.373 Low=0.490 lower
    x2: High=0.597 Low=0.488 higher
    x3: High=0.639 Low=0.419 higher
    x4: High=0.718 Low=0.356 higher
  [Fix1] Grid: Trust-LHS r=0.08 4D 50,000pts


  SVM mask: 50,000 => 50,000 (100%)
  GP: Matern(length_scale=0.356, nu=2.5)
  Sanity: pred=7.2800 actual=7.2800 std=0.002120


  UCB(b=0.15) max=7.6502 => [0.1673 0.8810 0.9789 0.9542]
  EI(xi=0.01) max=0.420168 => [0.1660 0.8757 0.9787 0.9724]
  Ensemble: UCB=7.5574 EI=7.5347 => UCB
  Dist from best: 0.1205

  >>> SUBMIT F5: 0.167299-0.881015-0.978872-0.954244 <<<
  [Fix5] Manual fine-tune baseline: [0.241041 0.805036 0.948951 0.90509 ]
  [Fix5] Fine-tune search: 50,000 micro-perturbations around W1 best


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# F6 — Cake Recipe (5D)
# W3: -2.509 — massive regression from W1 best (-0.361).
# Strategy: trust region around W1 best X. Low beta + EI to beat -0.361.
# ─────────────────────────────────────────────────────────────────────────────
next_x6, portal6 = analyse_function_w4(
    func_num     = 6,
    beta_ucb     = 1.5,     # reduced — return to proven region
    use_ei       = True,
    use_svm_mask = True,
    use_lhs      = True,    # LHS within trust region for 5D
    trust_radius = 0.20,
    xi           = 0.001,   # small xi — outputs negative but small magnitude
)


F6 Cake Recipe | Dim=5 N=23 BestY=-3.6118e-01
Best X* = [0.4670 0.3569 0.4897 0.7264 0.1251]
W3 = -2.5090e+00 (regressed)
  SVM threshold=-1.536e+00: High=12 Low=11
    x1: High=0.524 Low=0.550 similar
    x2: High=0.496 Low=0.666 lower
    x3: High=0.474 Low=0.433 similar
    x4: High=0.672 Low=0.383 higher
    x5: High=0.275 Low=0.618 lower
  [Fix1] Grid: Trust-LHS r=0.20 5D 50,000pts


  SVM mask: 50,000 => 50,000 (100%)
  GP: Matern(length_scale=0.554, nu=2.5)
  Sanity: pred=1.0184 actual=1.0184 std=0.000429
  UCB(b=1.5) max=1.2643 => [0.2909 0.2773 0.4929 0.7712 0.0241]
  EI(xi=0.001) max=0.066090 => [0.3346 0.2939 0.5008 0.7698 0.0749]
  Ensemble: UCB=0.9897 EI=1.0426 => EI
  Dist from best: 0.1612

  >>> SUBMIT F6: 0.334649-0.293944-0.500782-0.769829-0.074923 <<<


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# F7 — ML Hyperparameter Tuning (6D)
# W3: 1.253 — regression from W1 best (1.406). Small gap, recoverable.
# Strategy: moderate trust radius + balanced beta + EI to beat 1.406.
# ─────────────────────────────────────────────────────────────────────────────
next_x7, portal7 = analyse_function_w4(
    func_num     = 7,
    beta_ucb     = 2.0,     # moderate — steer toward W1 region
    use_ei       = True,
    use_svm_mask = True,
    use_lhs      = True,    # LHS essential for 6D
    trust_radius = 0.25,    # wider trust: 6D, best gap is only 0.153
    xi           = 0.01,
)


F7 ML Hyperparameters | Dim=6 N=33 BestY=1.4058e+00
Best X* = [0.0277 0.5318 0.3371 0.1761 0.3615 0.7308]
W3 = 1.2533e+00 (regressed)
  SVM threshold=9.264e-02: High=17 Low=16
    x1: High=0.377 Low=0.569 lower
    x2: High=0.368 Low=0.437 lower
    x3: High=0.398 Low=0.368 similar
    x4: High=0.427 Low=0.531 lower
    x5: High=0.319 Low=0.604 lower
    x6: High=0.551 Low=0.463 higher
  [Fix1] Grid: Trust-LHS r=0.25 6D 50,000pts


  SVM mask: 50,000 => 50,000 (100%)
  GP: Matern(length_scale=0.544, nu=2.5)
  Sanity: pred=0.3406 actual=0.3406 std=0.001887
  UCB(b=2.0) max=2.3101 => [0.0544 0.2870 0.1627 0.2592 0.1255 0.6233]
  EI(xi=0.01) max=0.466636 => [0.2064 0.2820 0.3894 0.2815 0.2188 0.7116]
  Ensemble: UCB=0.1746 EI=0.5947 => EI
  Dist from best: 0.3590

  >>> SUBMIT F7: 0.206363-0.281987-0.389442-0.281544-0.218827-0.711599 <<<


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# F8 — Complex 8D
# W3: 7.579 — regression from W1 best (9.892). Gap of 2.3.
# Strategy: moderate trust radius + LHS + EI to beat 9.892.
# ─────────────────────────────────────────────────────────────────────────────
next_x8, portal8 = analyse_function_w4(
    func_num     = 8,
    beta_ucb     = 2.0,
    use_ei       = True,
    use_svm_mask = True,
    use_lhs      = True,    # LHS critical for 8D
    trust_radius = 0.25,    # wider — 8D space needs more room
    xi           = 0.01,
)


F8 Complex 8D | Dim=8 N=43 BestY=9.8916e+00
Best X* = [0.1924 0.1831 0.0187 0.0364 0.6903 0.4442 0.0814 0.4290]
W3 = 7.5793e+00 (regressed)
  SVM threshold=7.924e+00: High=22 Low=21
    x1: High=0.370 Low=0.676 lower
    x2: High=0.418 Low=0.495 lower
    x3: High=0.356 Low=0.657 lower
    x4: High=0.361 Low=0.494 lower
    x5: High=0.466 Low=0.468 similar
    x6: High=0.477 Low=0.479 similar
    x7: High=0.495 Low=0.652 lower
    x8: High=0.509 Low=0.519 similar


  [Fix1] Grid: Trust-LHS r=0.25 8D 50,000pts


  SVM mask: 50,000 => 50,000 (100%)
  GP: Matern(length_scale=1.22, nu=2.5)
  Sanity: pred=2.2917 actual=2.2917 std=0.000128


  UCB(b=2.0) max=2.3904 => [0.0550 0.3842 0.0321 0.2641 0.7230 0.3824 0.2612 0.6724]
  EI(xi=0.01) max=0.027400 => [0.0955 0.3272 0.0513 0.2695 0.5558 0.4175 0.2851 0.6139]
  Ensemble: UCB=2.3183 EI=2.3257 => EI
  Dist from best: 0.4244

  >>> SUBMIT F8: 0.095545-0.327238-0.051339-0.269531-0.555763-0.417489-0.285113-0.613881 <<<


---
## Week 4 Final Submission Summary


In [14]:
portals = {
    1: portal1, 2: portal2, 3: portal3, 4: portal4,
    5: portal5, 6: portal6, 7: portal7, 8: portal8,
}

w4_betas = {1: 3.5, 2: 1.5, 3: 1.5, 4: 3.5, 5: 0.15, 6: 1.5, 7: 2.0, 8: 2.0}
w4_trust = {1: 'None', 2: '0.20', 3: '0.20', 4: 'None',
            5: '0.10', 6: '0.20', 7: '0.25', 8: '0.25'}
w4_modes = {1: 'Wide explore', 2: 'Trust+EI', 3: 'Trust+EI', 4: 'Wide LHS',
            5: 'Tight exploit', 6: 'Trust+EI', 7: 'Trust+EI', 8: 'Trust+EI'}

# All-time best (across all weeks including initial)
alltime_best = {i: data[i]['Y'].max() for i in range(1, 9)}

print('WEEK 4 FINAL SUBMISSION SUMMARY')
print('=' * 110)
print(f'{"Fn":<4} {"Description":<24} {"Dim":<5} {"Beta":<7} {"Trust":<7} {"All-time Best":<16} {"W3 Result":<16} {"Mode":<15} Submit')
print('-' * 110)
for i in range(1, 9):
    dim = data[i]['X'].shape[1]
    print(f'F{i:<3} {descriptions[i]:<24} {dim:<5} {w4_betas[i]:<7} {w4_trust[i]:<7} '
          f'{alltime_best[i]:<16.4e} {new_y_w3[i]:<16.4e} {w4_modes[i]:<15} {portals[i]}')
print('=' * 110)

print('\nCOPY-PASTE FOR PORTAL:')
print('-' * 50)
for i in range(1, 9):
    print(f'F{i}: {portals[i]}')

WEEK 4 FINAL SUBMISSION SUMMARY
Fn   Description              Dim   Beta    Trust   All-time Best    W3 Result        Mode            Submit
--------------------------------------------------------------------------------------------------------------
F1   Radiation Detection      2     3.5     None    2.8209e-04       2.6659e-174      Wide explore    0.278296-0.020000
F2   Noisy ML Model           2     1.5     0.20    6.1121e-01       -4.2551e-02      Trust+EI        0.685269-0.947006
F3   Drug Discovery           3     1.5     0.20    -1.0707e-02      -1.8409e-01      Trust+EI        0.403468-0.441923-0.497061
F4   Warehouse Placement      4     3.5     None    -3.4595e-01      -2.6070e+01      Wide LHS        0.352971-0.651614-0.805417-0.616108
F5   Chemical Yield (STAR)    4     0.15    0.10    1.4509e+03       1.1923e+03       Tight exploit   0.167299-0.881015-0.978872-0.954244
F6   Cake Recipe              5     1.5     0.20    -3.6118e-01      -2.5090e+00      Trust+EI        0

---
## Week 4 Reflection

### Why Week 3 Failed

Week 3 introduced SVM classification on just 12 data points. With so few observations:
- The SVM decision boundary was unreliable and may have mislabelled good regions as bad
- The GP with high beta values was attracted to unexplored high-uncertainty regions
- The combination pushed queries far from the historically proven-good X values

### Week 4 Core Change: Trust Regions

Rather than letting the GP explore freely, for 6 out of 8 functions we now constrain the search to a ball of radius r around the current best known X. This forces the algorithm to exploit the neighbourhood of what actually worked, not what the GP *thinks* might be good elsewhere.

| Function | Lesson |
|----------|--------|
| F2 | W3 went negative (from +0.171) — trust region prevents this |
| F3 | W3 was 17x worse — tight trust returns to W1 region |
| F5 | Star performer: 1450.94 → 1192 regression shows W3 drifted too far |
| F6 | W3 was 7x worse — clear overexploration penalty |
| F4 | No trust: still hasn't found a good region after 3 weeks |
| F1 | No trust: near-zero in all weeks, need wider search |

### Expected W4 Outcomes
- F2, F3, F6: should recover toward W1/W2 bests with trust + EI
- F5: trust radius 0.10 should find a point beating 1450.94 by fine-grained local search
- F7, F8: moderate trust should steer back toward W1 bests
- F4: remains uncertain — continued exploration
- F1: remains uncertain — 2D but function appears nearly flat everywhere
